# Step 3: Dataset Formatting & LLM Fine-Tuning

This notebook demonstrates the process of building the dataset using a teacher LLM (Gemini) and then fine-tuning a small on-device LLM (like Qwen or SmolLM).

## 1. Dataset Generation
Run the `dataset_builder.py` script. It will scan all `final_dialogue.json` files and use Gemini to extract structured CRM fields. It includes checkpointing, so you can stop and resume anytime.

In [8]:
!python pipeline/dataset_builder.py

Found 10 dialogue files. 10 already processed.
Building Dataset: 100%|██████████████████████| 10/10 [00:00<00:00, 52891.60it/s]
Dataset building complete. Added 0 new records to data/finetuning_dataset.jsonl


### Preview the generated dataset

In [9]:
import json
import pandas as pd

try:
    with open('data/finetuning_dataset.jsonl', 'r', encoding='utf-8') as f:
        # Load first 5 records
        data = [json.loads(next(f)) for _ in range(5)]
    df = pd.DataFrame(data)
    display(df)
except Exception as e:
    print("Run the dataset builder first to generate the dataset.", e)

,instruction,input,response
0,You are a structured-field extractor for teles...,Agent: à dạ a lô ạ à chị ạ em là hòa trương vi...,"{""customer_sector"": ""Bất động sản"", ""customer_..."
1,You are a structured-field extractor for teles...,Agent: a lô cho em hỏi đây cái số chị của anh\...,"{""customer_sector"": null, ""customer_needs"": ""i..."
2,You are a structured-field extractor for teles...,Customer: a lô\nAgent: vẻ luôn ạ dạ em chào ch...,"{""customer_sector"": null, ""customer_needs"": ""T..."
3,You are a structured-field extractor for teles...,Customer: a lô\nAgent: dạ em chào anh ạ anh có...,"{""customer_sector"": ""Retail and Dental clinic""..."
4,You are a structured-field extractor for teles...,Customer: a lô\nAgent: a lô à dạ vâng em chào ...,"{""customer_sector"": ""Bất động sản"", ""customer_..."


## 2. Model Fine-Tuning
Now we fine-tune a small LLM using LoRA via the `unsloth` library. You can choose different base models by passing the `--model` argument.
Examples:
- `unsloth/Qwen2.5-0.5B`
- `unsloth/Qwen1.5-0.5B`
- `unsloth/Llama-3-8b`
- `HuggingFaceTB/SmolLM2-1.7B-Instruct`

### Google Colab: Mount Drive
If running in Colab, mount your drive and CD to the project folder before training.

In [ ]:
# Run this if you are inside Google Colab
try:
    from google.colab import drive
    drive.mount("/content/drive")
    %cd "/content/drive/MyDrive/04 - University/04.02 - Adelaide University/04.02.02 - Sem 1 2026/02 - Deep Learning Applications/0 - Final Project/call-contextual-extractor"
except ImportError:
    print("Not running in Google Colab, skipping drive mount.")

In [10]:
!python pipeline/fine_tuner.py --model unsloth/Qwen2.5-0.5B

Loading model: unsloth/Qwen2.5-0.5B
Fetching 9 files: 100%|█████████████████████████| 9/9 [00:00<00:00, 3837.81it/s]
Download complete: : 0.00B [00:00, ?B/s]              Unsloth: Loading unsloth/Qwen2.5-0.5B via mlx-lm (runtime 4-bit affine quantization)...

Download complete: : 0.00B [00:00, ?B/s] [00:00, ?B/s]
Fetching 9 files: 100%|████████████████████████| 9/9 [00:00<00:00, 39819.34it/s]

Download complete: : 0.00B [00:00, ?B/s]              
[INFO] Quantized model with 7.671 bits per weight.
Unsloth: Quantized text model to 4-bit affine.
Unsloth: LoRA applied — 8,798,208 trainable params (4.38% of 200,914,816 total)
Generating train split: 10 examples [00:00, 753.92 examples/s]
Map: 100%|█████████████████████████████| 10/10 [00:00<00:00, 1183.86 examples/s]
Traceback (most recent call last):
  File "/Users/megatunger/Github/call-contextual-extractor/pipeline/fine_tuner.py", line 119, in <module>
    train_model(model_name=args.model, dataset_path=args.dataset)
  File "/Users/mega

## 3. Inference / Evaluation
Once the model is fine-tuned, you can load the generated LoRA weights to run inference on new transcripts.

In [ ]:
!python pipeline/inference.py --model data/finetuned_model_lora